# NB_hospital_daily_summary - Silver to Gold

Builds `<catalog>.gold.hospital_daily_summary` (one row per hospital per day) per the STTM `sttm_silver_to_gold.xlsx`. Driving table `silver.lab_results` grouped by hospital_id and test_date, LEFT JOINed to `hospitals` and the calendar dimension, with a LAG() day-over-day trend. Load type: FULL (CREATE OR REPLACE TABLE ... AS SELECT).

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_hospital.gold;

CREATE OR REPLACE TABLE adb_hospital.gold.hospital_daily_summary AS

WITH daily AS (
    SELECT
        hospital_id,
        test_date AS summary_date,
        COUNT(*) AS total_tests,
        COUNT(DISTINCT patient_id) AS unique_patients_tested,
        COUNT(DISTINCT doctor_id) AS active_doctor_count,
        COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
        COUNT_IF(result_status = 'Critical') AS critical_test_count

    FROM adb_hospital.silver.lab_results

    GROUP BY
        hospital_id,
        test_date
),

trend AS (
    SELECT
        *,
        LAG(total_tests) OVER (
            PARTITION BY hospital_id
            ORDER BY summary_date
        ) AS prev_day_total_tests

    FROM daily
)

SELECT
    t.hospital_id,
    h.hospital_name,
    t.summary_date,
    c.day_name,
    c.is_weekend,
    t.total_tests,
    t.unique_patients_tested,
    t.active_doctor_count,
    t.abnormal_test_count,
    t.critical_test_count,

    ROUND(
        t.abnormal_test_count * 100.0 / t.total_tests,
        2
    ) AS abnormal_rate_pct,

    ROUND(
        t.critical_test_count * 100.0 / t.total_tests,
        2
    ) AS critical_rate_pct,

    t.prev_day_total_tests,

    ROUND(
        (t.total_tests - t.prev_day_total_tests) * 100.0
        / NULLIF(t.prev_day_total_tests, 0),
        2
    ) AS test_volume_change_pct

FROM trend t

LEFT JOIN adb_hospital.silver.hospitals h
    ON t.hospital_id = h.hospital_id

LEFT JOIN adb_hospital.silver.calender c
    ON c.date_key = CAST(
        DATE_FORMAT(t.summary_date, 'yyyyMMdd') AS INT
    );

In [0]:
total_count = spark.table("adb_hospital.gold.hospital_daily_summary").count()
dbutils.notebook.exit(str(total_count))